In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install spark-nlp pyspark nltk spark-nlp-display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.8/758.8 kB 11.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.9/66.9 kB 4.3 MB/s eta 0:00:00


In [ ]:
#!pip install johnsnowlabs

In [3]:
import sparknlp
#import sparknlp_jsl

# Import Spark NLP
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline
from sparknlp_display import NerVisualizer

import pyspark.sql.functions as F

import warnings

In [4]:
# Settings and parameters for the Spark session
# As included in JSL notebooks

warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"16G", 
          "spark.kryoserializer.buffer.max":"2000M", 
          "spark.driver.maxResultSize":"2000M"} 

print("Spark NLP Version :", sparknlp.version())
#print("Spark NLP_JSL Version :", sparknlp_jsl.version())

# Staring Healthcare Spark NLP session
# spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
#spark = sparknlp_jsl.start(params=params)
spark = sparknlp.start(params=params)
spark

Spark NLP Version : 6.3.3
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0993445b-69fd-4cca-8b25-1aa449dfcbe7;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.g

In [5]:
sample_text = """John Doe called on 2025-12-05 at 555-123-4567. On September 20, 2025, I went to the Emergency Department at the Jewish General Hospital \
due to a sharp pain on my right side. Note: Since the doctor’s name is not mentioned in my visit report, here are some specific details from my visit that \
may help you identify the professionals involved so that you can better address patient needs in the future. I personally only go to the emergency department \
when I am truly in severe pain and unable to sleep because of it, as the long wait times make hospital visits very difficult. I assume that nobody would go to \
the emergency department for fun. Room: RAZWR3 Bed: 289 EDBK BLEU PAVK Id# DI. EMERGENCY, DEPARTMENT License# Unions representing workers at Turner Newall say \
they are 'disappointed' after talks with stricken parent firm Federal Mogul. TORONTO, Canada A second team of rocketeers competing for the #36;10 million \
Ansari X Prize. A company founded by a chemistry researcher at the University of Louisville won a grant to develop a method of producing better peptides. \
It's barely dawn when Mike Fitzpatrick starts his shift with a blur of colorful maps, figures and endless charts, but already he knows what the day will bring. \
Southern California's smog fighting agency went after emissions of the bovine variety Friday."""

# Visualization (pre-trained model)

In [6]:
# Define a pipeline
pipeline = PretrainedPipeline('recognize_entities_dl', lang='en')

recognize_entities_dl download started this may take some time.
Approx size to download 159 MB
[ | ]

26/03/22 18:46:56 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/22 18:46:56 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


recognize_entities_dl download started this may take some time.
Approximate size to download 159 MB
[ — ]Download done! Loading the resource.
[ \ ]

[OK!]


In [7]:
ppres = pipeline.fullAnnotate(sample_text)[0]

In [8]:
visualiser = NerVisualizer()

In [9]:
visualiser.display(ppres, label_col='entities', document_col='document', save_path=f"display_recognize_entities.html")

In [11]:
# Define the color codes for certain entities
visualiser.set_label_colors({'LOC':'#008080', 'PER':'#800080'})

visualiser.display(ppres, label_col='entities')

In [12]:
# Filter the entities

visualiser.display(ppres, label_col='entities', document_col='document',
                   labels=['PER'])

print()
print ('color code for label "PER": ' + visualiser.get_label_color('PER'))


color code for label "PER": #800080


In [13]:
# Step 1: Transforms raw texts to `document` annotation
document = DocumentAssembler() \
.setInputCol("text") \
.setOutputCol("document")

In [14]:
# Step 2: Tokenization
token = Tokenizer() \
.setInputCols("document") \
.setOutputCol("token") 

In [15]:
# Step 3: Bert Embeddings
embeddings = DistilBertEmbeddings.pretrained('distilbert_base_cased', 'en') \
.setInputCols(["document", "token"]) \
.setOutputCol("embeddings")

# Step 3: Get the embeddings using glove_100d
#embeddings = WordEmbeddingsModel.pretrained('glove_100d').\
#                  setInputCols(["document", 'token']).\
#                  setOutputCol("embeddings")

distilbert_base_cased download started this may take some time.


26/03/22 18:50:38 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 232.5 MB
[ | ]

26/03/22 18:50:39 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/22 18:50:39 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


distilbert_base_cased download started this may take some time.
Approximate size to download 232.5 MB
[ \ ]Download done! Loading the resource.
Using CPUs
[OK!]


In [16]:
# Step 4: Entity Extraction
ner_model= NerDLModel.pretrained("ner_ontonotes_distilbert_base_cased", 'en') \
.setInputCols(["document", "token", "embeddings"]) \
.setOutputCol("ner")

# Step 4: Use the ner_dl model
#public_ner = NerDLModel.pretrained('ner_dl', 'en') \
#          .setInputCols(["document", "token", "embeddings"]) \
#          .setOutputCol("ner")

ner_ontonotes_distilbert_base_cased download started this may take some time.


26/03/22 18:51:05 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 15.7 MB
[ | ]

26/03/22 18:51:06 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/22 18:51:06 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_ontonotes_distilbert_base_cased download started this may take some time.
Approximate size to download 15.7 MB
[ — ]Download done! Loading the resource.
[OK!]


`NerConverter` is used to convert the output of a NER model in Spark NLP into a format that is compatible with other downstream machine learning pipelines or applications.

`NerConverter` converts a **IOB** format (short for `inside`, `outside`, `beginning`) or IOB2 representation of NER to a user-friendly one, by associating the tokens of recognized entities and their label.

`NerConverter` expects `DOCUMENT`, `NAMED_ENTITY` and `TOKEN` as input, and then will provide `CHUNK` as output. The previous steps are expected to generate those annotator types, so that they will be used as input to `NerConverter`.

In [17]:
# Step 5: Convert to user-friendly NER
ner_converter = NerConverter() \
.setInputCols(["document", "token", "ner"]) \
.setOutputCol("entities")

In [18]:
# Define the pipeline
nlpPipeline = Pipeline(
    stages=[
        document, 
        token,
        embeddings,
        ner_model,
        ner_converter
])

#nlpPipeline = Pipeline(stages=[ documentAssembler, 
#                                 tokenizer,
#                                 embeddings,
#                                 public_ner,
#                                 ner_converter
#                                 ])

In [19]:
# Create the dataframe
empty_df = spark.createDataFrame([['']]).toDF("text")

In [20]:
# Fit the dataframe to the pipeline to get the model
pipelineModel = nlpPipeline.fit(empty_df)

In [21]:
# Convert the text to Dataframe
sample_data = spark.createDataFrame([[sample_text]]).toDF("text")

In [22]:
# Convert to LightPipeline Model
lmodel = LightPipeline(pipelineModel)

In [23]:
lresult = lmodel.annotate(sample_text)

Here, **B-** stands for the beginning token for an extracted entity, **I-** stands for a token coming second or later after the beginning token and **O** is a label for any other entity which is not defined as an NER in that model.

In [24]:
print(list(zip(lresult['token'], lresult['ner'])))

[('John', 'B-PERSON'), ('Doe', 'I-PERSON'), ('called', 'O'), ('on', 'O'), ('2025-12-05', 'B-DATE'), ('at', 'O'), ('555-123-4567', 'B-CARDINAL'), ('.', 'O'), ('On', 'O'), ('September', 'B-DATE'), ('20', 'I-DATE'), (',', 'I-DATE'), ('2025', 'I-DATE'), (',', 'O'), ('I', 'O'), ('went', 'O'), ('to', 'O'), ('the', 'B-ORG'), ('Emergency', 'I-ORG'), ('Department', 'I-ORG'), ('at', 'O'), ('the', 'B-ORG'), ('Jewish', 'I-ORG'), ('General', 'I-ORG'), ('Hospital', 'I-ORG'), ('due', 'O'), ('to', 'O'), ('a', 'O'), ('sharp', 'O'), ('pain', 'O'), ('on', 'O'), ('my', 'O'), ('right', 'O'), ('side', 'O'), ('.', 'O'), ('Note', 'O'), (':', 'O'), ('Since', 'O'), ('the', 'O'), ('doctor’s', 'O'), ('name', 'O'), ('is', 'O'), ('not', 'O'), ('mentioned', 'O'), ('in', 'O'), ('my', 'O'), ('visit', 'O'), ('report', 'O'), (',', 'O'), ('here', 'O'), ('are', 'O'), ('some', 'O'), ('specific', 'O'), ('details', 'O'), ('from', 'O'), ('my', 'O'), ('visit', 'O'), ('that', 'O'), ('may', 'O'), ('help', 'O'), ('you', 'O'), ('i

In [25]:
# Fit the pipeline to get the model 
nlpPipelineModel = nlpPipeline.fit(sample_data)

In [26]:
result = nlpPipelineModel.transform(sample_data)

In [ ]:
#result.select(F.explode(F.arrays_zip(result.entities.result, 
#                                     result.entities.metadata)).alias("cols")).show(truncate=False)

In [27]:
result.select(F.explode(F.arrays_zip(result.entities.result, 
                                     result.entities.metadata)).alias("cols")) \
      .select(F.expr("cols['1']['sentence']").alias("sentence_id"),
              F.expr("cols['0']").alias("word_chunk"),
              F.expr("cols['1']['entity']").alias("ner_label"),
              F.expr("cols['1']['confidence']").alias("confidence_level")).show(truncate=False)

+-----------+----------------------------+---------+----------------+
|sentence_id|word_chunk                  |ner_label|confidence_level|
+-----------+----------------------------+---------+----------------+
|0          |John Doe                    |PERSON   |0.91185         |
|0          |2025-12-05                  |DATE     |0.5874          |
|0          |555-123-4567                |CARDINAL |0.2916          |
|0          |September 20, 2025          |DATE     |0.66777503      |
|0          |the Emergency Department    |ORG      |0.7679          |
|0          |the Jewish General Hospital |ORG      |0.48974997      |
|0          |DEPARTMENT License# Unions  |ORG      |0.58883333      |
|0          |Turner Newall               |ORG      |0.42444998      |
|0          |Federal Mogul               |ORG      |0.6083          |
|0          |TORONTO                     |GPE      |0.8122          |
|0          |Canada A                    |GPE      |0.7981          |
|0          |second 

The main purpose of `NerOverwriter` is to modify the output of a pre-existing NER model to identify and annotate additional entities or to correct false positive or false negative annotations.

In [28]:
nerOverwriter = NerOverwriter() \
    .setInputCols(["ner"]) \
    .setOutputCol("ner_overwritten") \
    .setNerWords(["Unions"]) \
    .setNewNerEntity("B-ORG")

nerOverwriter.transform(result).selectExpr("explode(ner_overwritten)").show(truncate=False)

+---------------------------------------------------------------------------------------------------+
|col                                                                                                |
+---------------------------------------------------------------------------------------------------+
|{named_entity, 0, 3, B-PERSON, {word -> John, confidence -> 0.8791, sentence -> 0}, []}            |
|{named_entity, 5, 7, I-PERSON, {word -> Doe, confidence -> 0.9446, sentence -> 0}, []}             |
|{named_entity, 9, 14, O, {word -> called, confidence -> 0.9993, sentence -> 0}, []}                |
|{named_entity, 16, 17, O, {word -> on, confidence -> 0.9478, sentence -> 0}, []}                   |
|{named_entity, 19, 28, B-DATE, {word -> 2025-12-05, confidence -> 0.5874, sentence -> 0}, []}      |
|{named_entity, 30, 31, O, {word -> at, confidence -> 0.9614, sentence -> 0}, []}                   |
|{named_entity, 33, 44, B-CARDINAL, {word -> 555-123-4567, confidence -> 0.2916, s

# Visualization (BERT)

[Detect Entities (BERT)](http://sparknlp.org/2021/08/04/ner_ontonotes_distilbert_base_cased_en.html), where the model automatically extracts the following entities using `distilbert_base_casedembeddings`:

`CARDINAL`, `DATE`, `EVENT`, `FAC`, `GPE`, `LANGUAGE`, `LAW`, `LOC`, `MONEY`, `NORP`, `ORDINAL`, `ORG`, `PERCENT`, `PERSON`, `PRODUCT`, `QUANTITY`, `TIME`, `WORK_OF_ART`

The model was trained using `DistilBertEmbeddings` (`distilbert_base_cased`), so same embeddings must be used in the pipeline.

In [29]:
# Full annotate the light model to get predictions
cpres = lmodel.fullAnnotate(sample_text)

In [ ]:
#print(cpres)

In [30]:
# Define the color codes for certain entities
visualiser.set_label_colors({'LOC':'#008080', 'PERSON':'#800080', 'DATE':'#080080', 'ORG':'#088088', 'CARDINAL':'#008008', 'TIME':'#000000'})

visualiser.display(cpres[0], label_col='entities', document_col='document', save_path=f"display_result_ner.html")

In [31]:
visualiser.display(cpres[0], label_col='entities', document_col='document',
                   labels=['PERSON'])

In [32]:
visualiser.display(cpres[0], label_col='entities', document_col='document',
                   labels=['ORG'])

In [33]:
visualiser.display(cpres[0], label_col='entities', document_col='document',
                   labels=['DATE', 'TIME'])

In [34]:
visualiser.display(cpres[0], label_col='entities', document_col='document',
                   labels=['CARDINAL'])

In [35]:
visualiser.display(cpres[0], label_col='entities', document_col='document',
                   labels=['GPE'])